In [21]:
import pandas as pd
from collections import defaultdict, Counter


In [22]:
clinical_included_ids_file = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/06_preclin_clinic_join/data/clinical/clinical_inlcuded_18609_nctids.csv"
included_nctids = pd.read_csv(clinical_included_ids_file)


## code for stats

In [23]:
import pandas as pd

def add_unique_entity_count(
    df: pd.DataFrame,
    col: str,
    *,
    sep: str = "|",
    case_insensitive: bool = True,
    new_col: str | None = None,
) -> pd.DataFrame:
    """
    Add a column counting unique entities per row from a pipe-separated column.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    col : str
        Column containing pipe-separated entities.
    sep : str, default "|"
        Separator used in the column.
    case_insensitive : bool, default True
        Whether to normalize entities to lowercase before counting.
    new_col : str, optional
        Name of the new column. Defaults to `{col}_unique_count`.

    Returns
    -------
    pd.DataFrame
        Copy of dataframe with the added unique-count column.
    """
    df = df.copy()
    out_col = new_col or f"{col}_unique_count"

    def count_unique(val):
        if pd.isna(val):
            return 0
        items = [
            v.strip().lower() if case_insensitive else v.strip()
            for v in str(val).split(sep)
            if v.strip()
        ]
        return len(set(items))

    df[out_col] = df[col].apply(count_unique)
    return df


In [24]:
def no_linked_entities(termids: str) -> bool:
    """
    Return True if NONE of the entities were linked.
    Condition:
      - all values == '-1'
      - or empty / None / whitespace
    """
    if not termids:  # None or empty string
        return True
    
    # Split and clean
    parts = [p.strip() for p in str(termids).split("|") if p.strip()]

    if not parts:
        return True  # nothing there → no linked entities

    # True only if every part is '-1'
    return all(p == "-1" for p in parts)


In [25]:

def clean_pipe_split(s: str, sep: str = "|"):
    """Split pipe string, strip, drop empty tokens."""
    return [p.strip() for p in str(s or "").split(sep) if p and str(p).strip()]

def unique_terms_set(series: pd.Series, sep: str = "|", case_insensitive: bool = True) -> set:
    out = set()
    for val in series.fillna(""):
        for t in str(val).split(sep):
            t = t.strip()
            if not t:
                continue
            out.add(t.lower() if case_insensitive else t)
    return out


In [26]:
def summarize_entity_mappings(
    df,
    norm_col: str,
    id_col: str,
    ner_col: str,
    doc_id_col: str = "PMID",
    ignore_id: str = "-1",
):
    unique_entities_with_mappings = {}   # normalized entity -> first seen mapping id
    entity_frequency = defaultdict(int)  # normalized entity -> count
    mapped_to_ner = defaultdict(set)     # "entity (id)" -> set of original ner tokens

    for _, row in df.iterrows():
        doc_id = row.get(doc_id_col, "")

        norm_vals = clean_pipe_split(row.get(norm_col))
        id_vals   = clean_pipe_split(row.get(id_col))
        ner_vals  = clean_pipe_split(row.get(ner_col))

        # names vs ids must align
        if len(norm_vals) != len(id_vals):
            print(f"ERROR (names vs mappings length) {doc_id_col} {doc_id}")
            print(f"  {norm_col}: {norm_vals}")
            print(f"  {id_col}  : {id_vals}")
            continue

        for entity, mapping in zip(norm_vals, id_vals):
            entity = entity.strip()
            mapping = mapping.strip()
            if not entity:
                continue

            unique_entities_with_mappings.setdefault(entity, mapping)
            entity_frequency[entity] += 1

        # alignment to NER is optional; log if mismatch
        if len(norm_vals) != len(ner_vals):
            print(
                f"NOTE (names vs NER length) {doc_id_col} {doc_id}:\n"
                f"  → {len(norm_vals)} names vs {len(ner_vals)} NER tokens\n"
                f"  names: {norm_vals}\n"
                f"  ner  : {ner_vals}\n"
            )
            continue

        # build mapped_to_ner only for valid mappings
        for entity, mapping, ner_token in zip(norm_vals, id_vals, ner_vals):
            mapping = mapping.strip()
            ner_token = ner_token.strip()
            if mapping != ignore_id and ner_token:
                key = f"{entity} ({mapping})"
                mapped_to_ner[key].add(ner_token)

    mapped_to_ner_df = pd.DataFrame([
        {
            "entity_mapping": key,
            "entity": key.split("(")[0].strip(),
            "mapping": key.split("(")[1].rstrip(")").strip(),
            "ner_tokens": sorted(values),
            "ner_count": len(values),
        }
        for key, values in mapped_to_ner.items()
    ]).sort_values(by="ner_count", ascending=False).reset_index(drop=True)

    return unique_entities_with_mappings, entity_frequency, mapped_to_ner, mapped_to_ner_df


In [27]:
def entity_linking_report(
    df: pd.DataFrame,
    *,
    norm_col: str,
    id_col: str,
    ner_col: str,
    parent_label_col: str,
    clean_label_col: str | None = None, 
    doc_id_col: str = "PMID",
    ignore_id: str = "-1",
    sep: str = "|",
    case_insensitive: bool = True,
    round_digits: int = 2,
):

    # detailed summary on normalized entities
    unique_norm_to_id, norm_freq, mapped_to_ner, mapped_to_ner_df = summarize_entity_mappings(
        df=df,
        norm_col=norm_col,
        id_col=id_col,
        ner_col=ner_col,
        doc_id_col=doc_id_col,
        ignore_id=ignore_id,
    )

    # unique counts (raw, linked, linked+parent)
    raw_set = unique_terms_set(df[ner_col], sep=sep, case_insensitive=case_insensitive)
    linked_set = unique_terms_set(df[norm_col], sep=sep, case_insensitive=case_insensitive)
    parent_set = unique_terms_set(df[parent_label_col], sep=sep, case_insensitive=case_insensitive)

    count_unique_raw = len(raw_set)
    count_unique_linked = len(linked_set)
    count_unique_with_parent = len(parent_set)
    
    if clean_label_col is not None and clean_label_col in df.columns:
        clean_set = unique_terms_set(
            df[clean_label_col], sep=sep, case_insensitive=case_insensitive
        )
        count_unique_clean = len(clean_set)
    else:
        count_unique_clean = None

    # --- linking success computed from RAW tokens aligned with IDs ---
    raw_freq = Counter()     # raw -> mentions
    raw_to_first_id = {}     # raw -> first id at same position
    mismatch_rows = 0

    for _, row in df.iterrows():
        raw_vals = clean_pipe_split(row.get(ner_col), sep=sep)
        id_vals  = clean_pipe_split(row.get(id_col), sep=sep)

        if len(raw_vals) != len(id_vals):
            mismatch_rows += 1

        for raw, mid in zip(raw_vals, id_vals):
            raw_key = raw.lower() if case_insensitive else raw
            raw_freq[raw_key] += 1
            raw_to_first_id.setdefault(raw_key, mid)

    mapped_unique_raw = sum(1 for mid in raw_to_first_id.values() if mid != ignore_id)
    
    total_mentions = sum(raw_freq.values())
    mapped_mentions = sum(
        freq for raw_key, freq in raw_freq.items()
        if raw_to_first_id.get(raw_key, ignore_id) != ignore_id
    )
    pct_mapped_raw = (mapped_unique_raw / total_mentions * 100) if total_mentions else 0.0
    pct_mapped_freq_adjusted = (mapped_mentions / total_mentions * 100) if total_mentions else 0.0

    # --- build report row ---
    report_row = {
        "count unique raw entities": count_unique_raw,
        "count unique linked entities": count_unique_linked,
        "count unique linked with parent": count_unique_with_parent,
        "Linking success (unique / % mapped freq adjusted)": (
            f"{mapped_unique_raw} ({round(pct_mapped_raw, round_digits)}%)/{count_unique_raw} "
            f"({round(pct_mapped_freq_adjusted, round_digits)}%)"
        ),
    }

    if count_unique_clean is not None:
        report_row["count unique clean labels"] = count_unique_clean

    report = pd.DataFrame([report_row])

    details = {
        "unique_norm_to_id": unique_norm_to_id,
        "norm_frequency": norm_freq,
        "mapped_to_ner": mapped_to_ner,
        "mapped_to_ner_df": mapped_to_ner_df,
        "raw_frequency": raw_freq,
        "raw_to_first_id": raw_to_first_id,
    }

    return report, details


In [28]:
def get_unique_terms(df, col):
    """
    Extract unique pipe-separated terms (case-insensitive).
    Returns a sorted list of unique lowercase cleaned terms.
    """
    unique_terms = set()
    
    for val in df[col].fillna(""):
        terms = [
            t.strip().lower()
            for t in str(val).split("|")
            if t.strip()
        ]
        unique_terms.update(terms)
    
    return sorted(unique_terms)


In [29]:
def top_mapped_unmapped_side_by_side(
    unique_entities_with_mappings: dict,
    entity_frequency: dict,
    *,
    entity_name: str = "entity",
    unmapped_id: str = "-1",
    top_n: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Build a frequency table for entities, then return top-N mapped and unmapped
    entities side-by-side.

    Parameters
    ----------
    unique_entities_with_mappings : dict
        entity -> mapping_id (first seen mapping)
    entity_frequency : dict
        entity -> frequency / mentions
    entity_name : str
        Label used for the entity column in outputs (e.g. 'drug', 'disease')
    unmapped_id : str
        Value representing no mapping (default '-1')
    top_n : int
        Number of top mapped/unmapped to return

    Returns
    -------
    entity_df : DataFrame
        Full entity table: [entity, mapping, frequency, is_mapped]
    mapped_top : DataFrame
        Top-N mapped entities by frequency
    unmapped_top : DataFrame
        Top-N unmapped entities by frequency
    side_by_side : DataFrame
        Two tables concatenated horizontally: mapped vs unmapped
    """
    # Full table
    entity_df = pd.DataFrame([
        {
            entity_name: ent,
            "mapping": unique_entities_with_mappings.get(ent),
            "frequency": int(entity_frequency.get(ent, 0)),
        }
        for ent in unique_entities_with_mappings.keys()
    ])

    entity_df["is_mapped"] = entity_df["mapping"].astype(str).str.strip().ne(unmapped_id)

    entity_df = entity_df.sort_values(by="frequency", ascending=False).reset_index(drop=True)

    mapped_top = (
        entity_df[entity_df["is_mapped"]]
        .sort_values(by="frequency", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    unmapped_top = (
        entity_df[~entity_df["is_mapped"]]
        .sort_values(by="frequency", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    side_by_side = pd.concat(
        [
            mapped_top[[entity_name, "frequency"]].rename(
                columns={entity_name: f"Mapped {entity_name.title()}", "frequency": "Mapped Freq"}
            ),
            unmapped_top[[entity_name, "frequency"]].rename(
                columns={entity_name: f"Unmapped {entity_name.title()}", "frequency": "Unmapped Freq"}
            ),
        ],
        axis=1,
    )

    return entity_df, mapped_top, unmapped_top, side_by_side


In [30]:
def summarize_entities_per_document(
    df: pd.DataFrame,
    col: str,
    round_digits: int = 2,
):
    """
    Compute mean and standard deviation of unique entities per document.

    Returns
    -------
    dict with mean, std, min, max
    """
    return {
        "mean": round(df[col].mean(), round_digits),
        "std": round(df[col].std(), round_digits),
        "median": round(df[col].median(), round_digits),
        "min": int(df[col].min()),
        "max": int(df[col].max()),
    }


## Drug Entities

In [31]:
clinical_drugs="/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_clinical_drug_data_with_umls_parents.csv"
preclinical_drugs="/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_preclinical_drug_data_with_umls_parents.csv"

In [32]:
df_drugs_cli = pd.read_csv(clinical_drugs)
df_drugs_cli = df_drugs_cli[df_drugs_cli['nct_id'].isin(included_nctids['nct_id'])]
df_drugs_cli.shape

(18609, 11)

In [33]:
df_drugs_pre = pd.read_csv(preclinical_drugs)
df_drugs_pre.shape

(540999, 11)

### preclinical

In [34]:
doc_id_col = "PMID"
drug_raw_entities_col = "unique_interventions_linkbert_predictions"

drug_linked_entities_col = "drug_umls_term_norm"
drug_linked_entities_ids = "drug_umls_termid"

drug_parent_entities_col = "nearest_dataset_parent_umls_label"
drug_with_parents_entities_col = "merged_umls_label"
drug_with_parents_entities_ids = "merged_umls_termid"

cols_to_keep = [
    doc_id_col,
    drug_raw_entities_col,
    drug_linked_entities_col,
    drug_linked_entities_ids,
    drug_parent_entities_col,
    drug_with_parents_entities_col,
    drug_with_parents_entities_ids,
]

df_to_analyse = df_drugs_pre[cols_to_keep].copy()
df_to_analyse.head()

,PMID,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid
0,31733831,isorhynchophylline,isorhynchophylline,C0245133,-1,isorhynchophylline,C0245133
1,31733833,mir-1192|antgomir-1192|agomir-1192,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1,-1,"MIRN1192 microRNA, mouse|antgomir-1192|agomir-...",C3849422|-1|-1
2,31733925,hla-g2|g2,HLA-G2 Isoform|g2,C0967254|-1,-1,HLA-G2 Isoform|g2,C0967254|-1
3,31733940,minocycline,Minocycline,C0026187,-1,Minocycline,C0026187
4,31734027,tadalafil|phosphodiesterase type 5 inhibitor t...,Tadalafil|Tadalafil,C1176316|C1176316,-1,Tadalafil,C1176316


In [35]:
df_to_analyse["no_valid_mapping"] = (
    df_to_analyse["drug_umls_termid"].apply(no_linked_entities)
)
perc_no_mapping_at_all = df_to_analyse.no_valid_mapping.sum()/len(df_to_analyse)*100
df_to_analyse.no_valid_mapping.sum(), 100 - perc_no_mapping_at_all

(42797, 92.08926449032253)

In [36]:
df_to_analyse = add_unique_entity_count(
    df_to_analyse,
    col="merged_umls_label"
)
summary = summarize_entities_per_document(
    df_to_analyse,
    col="merged_umls_label_unique_count"
)

summary

{'mean': 2.7, 'std': 2.16, 'median': 2.0, 'min': 1, 'max': 44}

In [37]:
drug_report, drug_details = entity_linking_report(
    df=df_to_analyse,
    ner_col="unique_interventions_linkbert_predictions",  # raw
    norm_col="drug_umls_term_norm",                       # linked labels
    id_col="drug_umls_termid",                            # linked ids (aligned)
    parent_label_col="merged_umls_label",                 # linked + parent labels
    doc_id_col="PMID",
)
drug_report


,count unique raw entities,count unique linked entities,count unique linked with parent,Linking success (unique / % mapped freq adjusted)
0,426491,292485,292506,191149 (12.55%)/426491 (69.97%)


In [38]:
drug_details.keys()

dict_keys(['unique_norm_to_id', 'norm_frequency', 'mapped_to_ner', 'mapped_to_ner_df', 'raw_frequency', 'raw_to_first_id'])

In [39]:
mapped_drugs_to_ner_df = drug_details["mapped_to_ner_df"]
mapped_drugs_to_ner_df

,entity_mapping,entity,mapping,ner_tokens,ner_count
0,NG-Nitroarginine Methyl Ester (C0083536),NG-Nitroarginine Methyl Ester,C0083536,"[()-nitro-l-arginine methyl ester, (d)-nitro-l...",761
1,Glucagon-like peptide 1 receptor agonist (C156...,Glucagon-like peptide 1 receptor agonist,C1562104,"[(glp-1r) agonists, acting glp-2 analogs, acti...",346
2,"3 Inhibitor, Phosphodiesterase (C2267032)","3 Inhibitor, Phosphodiesterase",C2267032,"[(pde) inhibitors, 3'-5'-cyclic adenosine mono...",315
3,MK-801 (C0813872),MK-801,C0813872,"[(+ /-) mk-801, (+) mk-801, (+) mk801, (+))-mk...",294
4,Histone deacetylase inhibitor (C1512474),Histone deacetylase inhibitor,C1512474,[1 isoform-selective histone deacetylase inhib...,292
...,...,...,...,...,...
57165,KR-67607 (C5392614),KR-67607,C5392614,[kr-67607],1
57166,"cyclin-dependent kinase 12 protein, human (C16...","cyclin-dependent kinase 12 protein, human",C1620114,[cdk12],1
57167,norcocaethylene (C0655454),norcocaethylene,C0655454,[norcocaethylene],1
57168,"complement C3u protein, human (C1437780)","complement C3u protein, human",C1437780,[c3u],1


In [42]:
mapped_drugs_to_ner_df.to_csv("./data/linked_drugs_preclinical.csv", index=False)

In [40]:
drug_df, mapped_top10, unmapped_top10, side_by_side = top_mapped_unmapped_side_by_side(
    drug_details["unique_norm_to_id"],
    drug_details["norm_frequency"],
    entity_name="drug",
    unmapped_id="-1",
    top_n=10,
)

side_by_side


,Mapped Drug,Mapped Freq,Unmapped Drug,Unmapped Freq
0,NG-Nitroarginine Methyl Ester,6609,inhibitor,1308
1,Dexamethasone,6048,e2,1226
2,Acetylcysteine,5452,inhibitors,868
3,Cyclosporine,4995,of,677
4,Cisplatin,4474,antagonists,673
5,Sirolimus,4392,antagonist,577
6,Doxorubicin,4076,no donor,564
7,Melatonin,3856,selective serotonin reuptake inhibitor,540
8,Estradiol,3725,se,537
9,GLUCOCORTICOIDS,3688,ssris,517


### clinical

In [71]:
doc_id_col = "nct_id"

cols_to_keep = [
    doc_id_col,
    drug_raw_entities_col,
    drug_linked_entities_col,
    drug_linked_entities_ids,
    drug_parent_entities_col,
    drug_with_parents_entities_col,
    drug_with_parents_entities_ids,
]

df_to_analyse_cli = df_drugs_cli[cols_to_keep].copy()
df_to_analyse_cli.head()

,nct_id,unique_interventions_linkbert_predictions,drug_umls_term_norm,drug_umls_termid,nearest_dataset_parent_umls_label,merged_umls_label,merged_umls_termid
0,NCT03502551,ketamine,Ketamine,C0022614,-1,Ketamine,C0022614
1,NCT05216770,Laryngeal sensory block with topical bupivacaine,Laryngeal sensory block with topical bupivacaine,-1,-1,Laryngeal sensory block with topical bupivacaine,-1
2,NCT03348735,lidocaine|capsaicin|Lidocaine patch 5%,Lidocaine|Capsaicin|lidocaine 0.05 MG/MG Medic...,C0023660|C0006931|C0794811,-1,Lidocaine|Capsaicin|lidocaine 0.05 MG/MG Medic...,C0023660|C0006931|C0794811
3,NCT05995600,clopidogrel|warfarin|aspirin|Antiplatelet Drug,clopidogrel|Warfarin|Aspirin|Antiplatelet Drug,C0070166|C0043031|C0004057|-1,-1,clopidogrel|Warfarin|Aspirin|Antiplatelet Drug,C0070166|C0043031|C0004057|-1
4,NCT02137993,zyprexa|A-prexa,Zyprexa|A-prexa,C0527258|-1,olanzapine pamoate|OLANZapine|olanzapine pamoa...,Zyprexa|A-prexa|olanzapine pamoate|OLANZapine|...,C0527258|-1|C2698647|C0171023|C2726929


In [72]:
df_to_analyse_cli["no_valid_mapping"] = (
    df_to_analyse_cli["drug_umls_termid"].apply(no_linked_entities)
)
perc_not_linked = df_to_analyse_cli.no_valid_mapping.sum()/len(df_to_analyse_cli)*100
df_to_analyse_cli.no_valid_mapping.sum(), 100-perc_not_linked

(2439, 86.89343865871352)

In [73]:
df_to_analyse_cli = add_unique_entity_count(
    df_to_analyse_cli,
    col="merged_umls_label"
)
summary = summarize_entities_per_document(
    df_to_analyse_cli,
    col="merged_umls_label_unique_count"
)

summary

{'mean': 2.24, 'std': 1.59, 'median': 2.0, 'min': 1, 'max': 22}

In [74]:
drug_report, drug_details = entity_linking_report(
    df=df_to_analyse_cli,
    ner_col="unique_interventions_linkbert_predictions",  # raw
    norm_col="drug_umls_term_norm",                       # linked labels
    id_col="drug_umls_termid",                            # linked ids (aligned)
    parent_label_col="merged_umls_label",                 # linked + parent labels
    doc_id_col="nct_id",
)
drug_report


,count unique raw entities,count unique linked entities,count unique linked with parent,Linking success (unique / % mapped freq adjusted)
0,15453,12459,12717,8197 (19.97%)/15453 (74.11%)


In [75]:
drug_df, mapped_top10, unmapped_top10, side_by_side = top_mapped_unmapped_side_by_side(
    drug_details["unique_norm_to_id"],
    drug_details["norm_frequency"],
    entity_name="drug",
    unmapped_id="-1",
    top_n=10,
)

side_by_side


,Mapped Drug,Mapped Freq,Unmapped Drug,Unmapped Freq
0,PLACEBO,330,antidepressant,148
1,Levodopa,314,antipsychotic,106
2,Risperidone,299,antidepressants,74
3,Ketamine,253,antipsychotics,71
4,Dexmedetomidine,234,ssri,68
5,OLANZapine,227,selective serotonin reuptake inhibitor,44
6,ARIPiprazole,215,-,41
7,Botulinum Toxin Type A,201,ssris,39
8,"Fatty Acids, Omega-3",181,atypical antipsychotic,35
9,Cannabidiol,178,aeds,34


## Disease entities

In [43]:
clinical_disesases = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_clinical_data_mondo_cleaned_with_mondo_parents.csv"
preclinical_disesases = "/shares/animalwelfare.crs.uzh/Preclinical_Pipeline/04_normalization/data/mapped_all/mapped_preclinical_data_mondo_cleaned_with_mondo_parents.csv"

In [44]:
df_dis_cli = pd.read_csv(clinical_disesases)
df_dis_cli = df_dis_cli[df_dis_cli['nct_id'].isin(included_nctids['nct_id'])]
df_dis_cli.shape

(18609, 13)

In [45]:
df_dis_pre = pd.read_csv(preclinical_disesases)
df_dis_pre.shape

(540999, 13)

In [46]:
df_dis_pre.head()

,PMID,unique_conditions_linkbert_predictions,linkbert_mondo_predictions,disease_mondo_termid,disease_mondo_term_norm,disease_mondo_closest_3,disease_mondo_cdist,disease_term_mondo_clean,disease_termid_mondo_clean,nearest_dataset_parent_mondo,nearest_dataset_parent_label,merged_mondo_termid,merged_mondo_label
0,31733831,asthma,asthma,MONDO:0004979,asthma,"[['asthma', 'MONDO:0004979'], ['asthma, bronch...",0.0068,asthma,MONDO:0004979,-1,-1,MONDO:0004979,asthma
1,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,"[['myocardial infarction', 'MONDO:0005068'], [...",0.0,myocardial infarction,MONDO:0005068,-1,-1,MONDO:0005068,myocardial infarction
2,31733925,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915,systemic lupus erythematosus,"[['systemic lupus erythematosus', 'MONDO:00079...",0.0,systemic lupus erythematosus,MONDO:0007915,-1,-1,MONDO:0007915,systemic lupus erythematosus
3,31733940,cognitive impairment,cognitive disorder,MONDO:0002039,cognitive disorder,"[['cognitive disorder', 'MONDO:0002039'], ['co...",7.6573,cognitive disorder,MONDO:0002039,-1,-1,MONDO:0002039,cognitive disorder
4,31734027,cumulative peripheral neuropathy|oxaliplatin-i...,cumulative peripheral neuropathy|oxaliplatin-i...,-1|-1|MONDO:0003620,cumulative peripheral neuropathy|oxaliplatin-i...,"[['chronic acquired peripheral neuropathy', 'M...",9.8837|13.0604|0.0055,cumulative peripheral neuropathy|oxaliplatin-i...,-1|-1|MONDO:0003620,-1,-1,-1|-1|MONDO:0003620,cumulative peripheral neuropathy|oxaliplatin-i...


### preclinical

In [47]:
doc_id_col = "PMID"
disease_raw_entities_col = "unique_conditions_linkbert_predictions"

disease_linked_entities_col = "disease_mondo_term_norm"
disease_linked_entities_ids = "disease_mondo_termid"

cleaned_mondo_entities = "disease_term_mondo_clean"

disease_parent_entities_col = "nearest_dataset_parent_label"
disease_with_parents_entities_col = "merged_mondo_label"
disease_with_parents_entities_ids = "merged_mondo_termid"

cols_to_keep = [
    doc_id_col,
    disease_raw_entities_col,
    disease_linked_entities_col,
    disease_linked_entities_ids,
    cleaned_mondo_entities,
    disease_parent_entities_col,
    disease_with_parents_entities_col,
    disease_with_parents_entities_ids,
]

df_to_analyse = df_dis_pre[cols_to_keep].copy()
df_to_analyse.head()

,PMID,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid
0,31733831,asthma,asthma,MONDO:0004979,asthma,-1,asthma,MONDO:0004979
1,31733833,myocardial infarction,myocardial infarction,MONDO:0005068,myocardial infarction,-1,myocardial infarction,MONDO:0005068
2,31733925,systemic lupus erythematosus,systemic lupus erythematosus,MONDO:0007915,systemic lupus erythematosus,-1,systemic lupus erythematosus,MONDO:0007915
3,31733940,cognitive impairment,cognitive disorder,MONDO:0002039,cognitive disorder,-1,cognitive disorder,MONDO:0002039
4,31734027,cumulative peripheral neuropathy|oxaliplatin-i...,cumulative peripheral neuropathy|oxaliplatin-i...,-1|-1|MONDO:0003620,cumulative peripheral neuropathy|oxaliplatin-i...,-1,cumulative peripheral neuropathy|oxaliplatin-i...,-1|-1|MONDO:0003620


In [48]:
df_to_analyse["no_valid_mapping"] = (
    df_to_analyse["disease_mondo_termid"].apply(no_linked_entities)
)
perc_no_mapping_at_all = df_to_analyse.no_valid_mapping.sum()/len(df_to_analyse)*100
df_to_analyse.no_valid_mapping.sum(), 100 - perc_no_mapping_at_all

(70447, 86.97834931303015)

In [49]:
df_to_analyse = add_unique_entity_count(
    df_to_analyse,
    col="merged_mondo_label"
)
summary = summarize_entities_per_document(
    df_to_analyse,
    col="merged_mondo_label_unique_count"
)

summary

{'mean': 1.89, 'std': 1.18, 'median': 2.0, 'min': 1, 'max': 24}

In [50]:
disease_report, disease_details = entity_linking_report(
    df=df_to_analyse,
    ner_col=disease_raw_entities_col,  # raw
    norm_col=disease_linked_entities_col,                       # linked labels
    id_col=disease_linked_entities_ids,                            # linked ids (aligned)
    parent_label_col=disease_with_parents_entities_col,                 # linked + parent labels
    doc_id_col="PMID",
    clean_label_col="disease_term_mondo_clean",
)
disease_report

,count unique raw entities,count unique linked entities,count unique linked with parent,Linking success (unique / % mapped freq adjusted),count unique clean labels
0,123919,80349,79918,51357 (4.93%)/123919 (73.93%),79907


In [51]:
disease_df, mapped_top10, unmapped_top10, side_by_side = top_mapped_unmapped_side_by_side(
    disease_details["unique_norm_to_id"],
    disease_details["norm_frequency"],
    entity_name="disease",
    unmapped_id="-1",
    top_n=10,
)
side_by_side

,Mapped Disease,Mapped Freq,Unmapped Disease,Unmapped Freq
0,ischemia reperfusion injury,23708,hypoxia,4331
1,epilepsy,19368,liver,2650
2,Alzheimer disease,17855,stress,2445
3,diabetes mellitus,15451,oxidative stress,2296
4,ischemic disease,13451,focal cerebral ischemia,1904
5,Parkinson disease,11941,i/r injury,1673
6,depressive disorder,10105,liver injury,1667
7,cerebral infarction,8911,hyperalgesia,1612
8,stroke disorder,8329,septic shock,1417
9,neuralgia,8197,shock,1412


In [52]:
mapped_diseases_to_ner_df = disease_details["mapped_to_ner_df"]
mapped_diseases_to_ner_df


,entity_mapping,entity,mapping,ner_tokens,ner_count
0,ischemia reperfusion injury (MONDO:0005203),ischemia reperfusion injury,MONDO:0005203,"[(i))-reperfusion, / reperfusion injury, / rep...",1378
1,perinatal asphyxia (MONDO:0006663),perinatal asphyxia,MONDO:0006663,"[acute asphyxia, acute hypoxic-ischemic (hi) e...",467
2,brain hypoxia - ischemia (MONDO:0006685),brain hypoxia - ischemia,MONDO:0006685,"[acute brain hypoxia, acute cerebral hypoxia, ...",302
3,brain ischemia (MONDO:0005299),brain ischemia,MONDO:0005299,"[acute and chronic brain ischemia, acute and c...",207
4,type 2 diabetes mellitus (MONDO:0005148),type 2 diabetes mellitus,MONDO:0005148,"[1 and 2 diabetes, 1 and 2 diabetic, 2 diabete...",184
...,...,...,...,...,...
7783,perichondritis of auricle (MONDO:0002246),perichondritis of auricle,MONDO:0002246,[perichondritis of],1
7784,osteopenia-intellectual disability-sparse hair...,osteopenia-intellectual disability-sparse hair...,MONDO:0009814,[oestrogen-deficiency osteopaenia],1
7785,disorder of sphingolipid biosynthesis (MONDO:0...,disorder of sphingolipid biosynthesis,MONDO:0021130,[sphingolipid disorders],1
7786,mucopolysaccharidosis or mucopolysaccharidosis...,mucopolysaccharidosis or mucopolysaccharidosis...,MONDO:0100365,[mucopolysaccharidosis-like disease],1


In [53]:
mapped_diseases_to_ner_df.to_csv("./data/linked_diseases_preclinical.csv", index=False)

### clinical

In [92]:
doc_id_col = "nct_id"

cols_to_keep = [
    doc_id_col,
    disease_raw_entities_col,
    disease_linked_entities_col,
    disease_linked_entities_ids,
    cleaned_mondo_entities,
    disease_parent_entities_col,
    disease_with_parents_entities_col,
    disease_with_parents_entities_ids,
]

df_to_analyse = df_dis_cli[cols_to_keep].copy()
df_to_analyse.head()

,nct_id,unique_conditions_linkbert_predictions,disease_mondo_term_norm,disease_mondo_termid,disease_term_mondo_clean,nearest_dataset_parent_label,merged_mondo_label,merged_mondo_termid
0,NCT03502551,suicidal|suicidal ideation|Suicide,suicidal|suicidal ideation|Suicide,-1|-1|-1,suicidal|suicidal ideation|Suicide,-1,suicidal|suicidal ideation|Suicide,-1|-1|-1
1,NCT05216770,laryngeal dystonia|voice tremor|Tremor,spasmodic dystonia|voice tremor|obsolete rare ...,MONDO:0000485|-1|MONDO:0017644,spasmodic dystonia|voice tremor|obsolete rare ...,focal dystonia|-1,spasmodic dystonia|voice tremor|obsolete rare ...,MONDO:0000485|-1|MONDO:0017644|MONDO:0000477
2,NCT03348735,np|subacute|pain|localized neuropathic pain|ch...,np|subacute|obsolete disorder involving pain|l...,-1|-1|MONDO:0021668|-1|MONDO:0021667|MONDO:002...,np|subacute|obsolete disorder involving pain|l...,-1|-1,np|subacute|obsolete disorder involving pain|l...,-1|-1|MONDO:0021668|-1|MONDO:0021667
3,NCT05995600,systemic lupus|definite|antiphospholipid syndr...,systemic lupus erythematosus|definite|antiphos...,MONDO:0007915|-1|-1|MONDO:0005098|-1|MONDO:000...,systemic lupus erythematosus|definite|antiphos...,-1|-1|-1|-1|-1|brain ischemia,systemic lupus erythematosus|definite|antiphos...,MONDO:0007915|-1|-1|MONDO:0005098|-1|MONDO:000...
4,NCT02137993,schizophreniform disorder|schizoaffective diso...,schizophreniform disorder|schizophrenia|schizo...,MONDO:0001265|MONDO:0005090|MONDO:0005090,schizophreniform disorder|schizophrenia,-1|-1,schizophreniform disorder|schizophrenia,MONDO:0001265|MONDO:0005090


In [93]:
df_to_analyse["no_valid_mapping"] = (
    df_to_analyse["disease_mondo_termid"].apply(no_linked_entities)
)
perc_no_mapping_at_all = df_to_analyse.no_valid_mapping.sum()/len(df_to_analyse)*100
df_to_analyse.no_valid_mapping.sum(), 100 - perc_no_mapping_at_all

(379, 97.96335106668816)

In [96]:
df_to_analyse = add_unique_entity_count(
    df_to_analyse,
    col="merged_mondo_label"
)
summary = summarize_entities_per_document(
    df_to_analyse,
    col="merged_mondo_label_unique_count"
)

summary

{'mean': 2.96, 'std': 1.99, 'median': 2.0, 'min': 1, 'max': 31}

In [99]:
(123919 - 80349)/123919

0.3516006423550868

In [50]:
disease_report, disease_details = entity_linking_report(
    df=df_to_analyse,
    ner_col=disease_raw_entities_col,  # raw
    norm_col=disease_linked_entities_col,                       # linked labels
    id_col=disease_linked_entities_ids,                            # linked ids (aligned)
    parent_label_col=disease_with_parents_entities_col,                 # linked + parent labels
    doc_id_col="nct_id",
    clean_label_col="disease_term_mondo_clean",
)
disease_report

,count unique raw entities,count unique linked entities,count unique linked with parent,Linking success (unique / % mapped freq adjusted),count unique clean labels
0,15508,10916,10959,6644 (11.03%)/15508 (69.91%),10850


In [51]:
disease_df, mapped_top10, unmapped_top10, side_by_side = top_mapped_unmapped_side_by_side(
    disease_details["unique_norm_to_id"],
    disease_details["norm_frequency"],
    entity_name="disease",
    unmapped_id="-1",
    top_n=10,
)
side_by_side

,Mapped Disease,Mapped Freq,Unmapped Disease,Unmapped Freq
0,Alzheimer disease,1999,mild to moderate alzheimer's disease,205
1,schizophrenia,1917,smoking,184
2,depressive disorder,1780,fatigue,160
3,Parkinson disease,1713,mild cognitive impairment,146
4,bipolar disorder,1061,cocaine,134
5,stroke disorder,924,spasticity,108
6,epilepsy,915,agitation,101
7,migraine disorder,897,surgery,91
8,obsolete disorder involving pain,856,smokers,90
9,multiple sclerosis,736,postoperative delirium,75
